# Environment: episodios y pasos

Dos inputs → la misma unidad por paso → dos outputs e historia de episodio.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [ ]:
agent = toolkit.Agent("calculator", toolkit.Sequence((increase, multiply)), skills=(arithmetic,))
application = toolkit.System("team", toolkit.Sequence((agent,)))
episode = toolkit.Environment(name="two_steps", unit=application, records=({"value": 3}, {"value": -3}))
result = episode.run()
assert result.ok and result.data["outputs"] == [{"value": 8}, {"value": -4}]
assert result.data["summary"]["memory"] == {"processed": 2}
print({"input": [3, -3], "process": "un paso por registro", "output": result.data["outputs"]})

Input: importes 2, 3, 100. Proceso: observe combina registro y memoria; update conserva el total; stop termina al llegar a 5. Output: 2 y 5, sin procesar 100. La declaración se comparte con examples/episode_memory.

In [ ]:
from lesson_examples import build_episode
import inspect
print(inspect.getsource(build_episode))
episode = build_episode()
prepared = episode.compile()
result = prepared.run()
assert result.ok and result.data['summary']['memory'] == {'total':5}
assert result.data['summary']['terminated']
assert (await episode.arun()).data['outputs'] == result.data['outputs']
assert (await prepared.arun()).data['outputs'] == result.data['outputs']
print({'input':list(episode.records),'process':prepared.inspect(),'output':result.data['outputs']})

Cada run abre un episodio nuevo. Sin observe, cada registro es el input. observe y update conectan memoria real entre pasos; stop permite terminar el episodio. Las transiciones están en `result.data['history']` y las ejecuciones de las unidades en `result.children`.